In [1]:
# Parameters
BATCH_MODE = "true"


# PDF et Web Search : Sources Documentaires avec OpenAI

**Navigation** : [Index](README.md) | [<< Précédent](05_RAG_Modern.ipynb) | [Suivant >>](07_Code_Interpreter.ipynb)


Ce notebook explore deux fonctionnalités puissantes de l'API OpenAI :
- **Support PDF direct** : Envoyer des documents PDF aux modèles vision
- **Web Search** : Accéder à des informations en temps réel

**Objectifs :**
- Charger et analyser des PDFs via l'API
- Utiliser l'outil web_search pour la recherche en temps réel
- Combiner documents et recherche pour des réponses enrichies

**Prérequis :** Notebook 1 (OpenAI Intro)

**Durée estimée :** 50 minutes

In [2]:
# Installation des dépendances
from pathlib import Path
%pip install openai python-dotenv reportlab pillow -q

import os
import base64
from openai import OpenAI
from dotenv import load_dotenv

# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")
client = OpenAI()

# Charger le modèle depuis .env ou utiliser gpt-5-mini par défaut
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() == "true"

print("Client OpenAI initialisé !")
print(f"Modèle par défaut: {DEFAULT_MODEL}")
print(f"Mode: {'Batch' if BATCH_MODE else 'Interactive'}")


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


.env charge depuis: .env


Client OpenAI initialisé !
Modèle par défaut: gpt-5-mini
Mode: Interactive


## 1. Support de Documents dans l'API OpenAI

Les modèles OpenAI avec capacités vision peuvent traiter des images et, via l'Assistants API, des fichiers PDF :

**Modèles compatibles :**
- `gpt-4o-mini` et `gpt-4o` (avec vision)
- Tous les modèles vision de la famille GPT-4

**Approches pour les documents :**

| Approche | API | Avantages | Limites |
|----------|-----|-----------|---------|
| **Image directe** | Chat Completions | Simple, rapide | Uniquement images (png, jpg, gif, webp) |
| **Assistants + Files** | Assistants API | Supporte PDF natif | Plus complexe, coût stockage |
| **Conversion image** | Chat Completions | Universel | Perte de qualité potentielle |

**Note importante :** L'API Vision (Chat Completions) accepte uniquement les images, pas les PDF directement. Pour les PDF, utilisez l'Assistants API ou convertissez en images.

In [3]:
# Créer un document de test sous forme d'IMAGE (pour l'API Vision)
from PIL import Image, ImageDraw, ImageFont
import io

def create_test_image():
    """Génère une image représentant un rapport pour démonstration"""
    # Créer une image blanche A4-like (800x1000)
    img = Image.new('RGB', (800, 1000), color='white')
    draw = ImageDraw.Draw(img)
    
    # Utiliser une police par défaut
    try:
        font_title = ImageFont.truetype("arial.ttf", 36)
        font_normal = ImageFont.truetype("arial.ttf", 18)
        font_small = ImageFont.truetype("arial.ttf", 14)
    except:
        # Fallback si police non disponible
        font_title = ImageFont.load_default()
        font_normal = ImageFont.load_default()
        font_small = ImageFont.load_default()
    
    # Dessiner le contenu
    y = 50
    draw.text((100, y), "Rapport Trimestriel Q1 2026", fill='black', font=font_title)
    y += 80
    
    draw.text((100, y), "Résumé Exécutif", fill='darkblue', font=font_normal)
    y += 40
    
    lines = [
        "- Chiffre d'affaires: 2.5M EUR (+15%)",
        "- Nouveaux clients: 150 (+25%)",
        "- Satisfaction client: 4.5/5",
        "",
        "Points clés:",
        "1. Lancement réussi du produit Alpha",
        "2. Expansion sur le marché européen",
        "3. Recrutement de 20 ingénieurs",
        "",
        "Perspectives Q2: Objectif 3M EUR (+20%)"
    ]
    
    for line in lines:
        draw.text((100, y), line, fill='black', font=font_normal)
        y += 30
    
    # Sauvegarder en bytes
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    buffer.seek(0)
    return buffer.getvalue()

# Générer et sauvegarder l'image
img_content = create_test_image()
with open("test_report.png", "wb") as f:
    f.write(img_content)

print("✓ Image de rapport créée: test_report.png")
print(f"  Taille: {len(img_content)} octets")

✓ Image de rapport créée: test_report.png
  Taille: 40235 octets


### Pourquoi une image plutôt qu'un PDF ?

L'API **Chat Completions** avec vision (`gpt-5-mini`, `gpt-5-mini`) accepte uniquement des **images** (PNG, JPG, GIF, WebP), pas les PDF directement.

**Options pour traiter des PDFs :**

1. **Assistants API** : Support natif des PDF via l'upload de fichiers (plus complexe)
2. **Conversion PDF → Image** : Utiliser `pdf2image` ou `PyMuPDF` (méthode utilisée ici)
3. **Extraction texte** : Si le PDF contient du texte sélectionnable (`PyPDF2`, `pdfplumber`)

Dans ce notebook, nous créons une **image** qui simule un rapport pour démontrer l'analyse visuelle de documents.

In [4]:
# Analyser l'image via l'API OpenAI Vision
# Charger et encoder en base64
with open("test_report.png", "rb") as f:
    img_base64 = base64.b64encode(f.read()).decode()

print(f"Envoi de l'image au modèle {DEFAULT_MODEL}...\n")

# Envoyer au modèle avec vision.
# Note: gpt-5-mini est un modèle de raisonnement — les tokens de raisonnement sont
# déduits de max_completion_tokens. Un budget trop bas (ex. 500) est entièrement
# consommé par le raisonnement et laisse le contenu visible vide. Budget relevé à
# 4000 pour laisser de la place à une réponse réelle (#3571).
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "text", 
                "text": "Analyse ce rapport et donne-moi les 3 points clés avec les chiffres associés."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{img_base64}"
                }
            }
        ]
    }],
    max_completion_tokens=4000
)

print("=== Analyse du document ===")
print(response.choices[0].message.content)
print(f"\nTokens utilisés: {response.usage.total_tokens}")

Envoi de l'image au modèle gpt-5-mini...



=== Analyse du document ===
Voici les 3 points clés du rapport Q1 2026 avec les chiffres associés :

1) Chiffre d’affaires : 2,5 M EUR en Q1, +15% vs période précédente (objectif Q2 : 3 M EUR, +20%).  
2) Acquisition clients : 150 nouveaux clients en Q1, +25%.  
3) Ressources & satisfaction : recrutement de 20 ingénieurs ; satisfaction client : 4,5/5.

Souhaitez‑vous un bref commentaire stratégique pour chacun ?

Tokens utilisés: 1612


### Interprétation des résultats

Le modèle vision extrait correctement les 3 points clés du rapport, avec leurs
chiffres : chiffre d'affaires (2,5 M EUR, +15%), nouveaux clients (150, +25%) et
recrutement (20 ingénieurs). L'image 800×1000 est traitée en tokens visuels
(1958 tokens au total sur cette exécution).

**Capacités attendues d'un modèle vision sur un document lisible :**

1. **Reconnaissance optique** : analyse de l'image et extraction du texte visible
2. **Compréhension sémantique** : identification des sections (titre, résumé, points clés)
3. **Extraction ciblée** : sélection des informations principales avec leurs valeurs

**Limites potentielles (à l'origine d'une extraction vide) :**

- **Qualité image** : basse résolution ou flou dégradent la précision
- **Complexité visuelle** : graphiques complexes peuvent être mal interprétés
- **OCR** : polices spéciales ou petites peuvent causer des erreurs

> **Note technique** : `gpt-5-mini` est un modèle de raisonnement — les tokens de
> raisonnement sont déduits de `max_completion_tokens`. Un budget trop bas est
> entièrement consommé par le raisonnement et laisse le contenu visible vide ;
> c'est pourquoi `max_completion_tokens=4000` est utilisé ici (#3571).

> **Référence** : les modèles vision-langage multimodaux qui combinent OCR et
> compréhension sémantique s'inspirent de travaux comme Flamingo
> (Alayrac et al. 2022, *Flamingo: a Visual Language Model for Few-Shot Learning*,
> arXiv:2204.14198).</cell>
>

### Exercice 1 : Analyser une image personnalisee avec l'API Vision

En vous basant sur l'exemple d'analyse de rapport ci-dessus, créez votre propre image de test (un graphique ou un tableau de données) et demandez au modèle d'en extraire les informations cles.

**Objectif** : Generer une image avec PIL contenant un tableau de données (ventes mensuelles), l'encoder en base64, et demander au modèle d'en extraire les valeurs et la tendance.

**Indices** :
- Utilisez `PIL.Image`, `PIL.ImageDraw` et `PIL.ImageFont` pour créer une image avec un tableau
- Dessinez un tableau avec 4 lignes : "Janvier: 120k", "Fevrier: 145k", "Mars: 160k", "Avril: 180k"
- Encodez l'image en base64 avec `base64.b64encode()`
- Envoyez au modèle avec le prompt : "Extrais les ventes mensuelles de cette image et identifie la tendance"

In [5]:
# Exercice 1 : Analyser une image personnalisee avec l'API Vision
# TODO etudiant : Creez une image avec un tableau de ventes et analysez-la

# Etape 1 : Creer une image avec PIL contenant un tableau de ventes
# from PIL import Image, ImageDraw, ImageFont
# import io, base64
#
# img = Image.new('RGB', (600, 300), 'white')
# draw = ImageDraw.Draw(img)
# draw.text((50, 30), "Ventes Mensuelles 2026", fill='black')
# draw.text((50, 80), "Janvier: 120k EUR", fill='black')
# draw.text((50, 120), "Fevrier: 145k EUR", fill='black')
# draw.text((50, 160), "Mars: 160k EUR", fill='black')
# draw.text((50, 200), "Avril: 180k EUR", fill='black')
#
# buffer = io.BytesIO()
# img.save(buffer, format='PNG')
# img_b64 = base64.b64encode(buffer.getvalue()).decode()

# Etape 2 : Envoyer l'image au modele Vision
# response_vision = client.chat.completions.create(
#     model=DEFAULT_MODEL,
#     messages=[{
#         "role": "user",
#         "content": [
#             {"type": "text", "text": "Extrais les ventes mensuelles de cette image et identifie la tendance."},
#             {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
#         ]
#     }],
#     max_completion_tokens=300
# )

# Etape 3 : Afficher le resultat
# print(response_vision.choices[0].message.content)

print("Exercice a completer")

Exercice a completer


## 2. Web Search avec l'API OpenAI

L'outil **web_search_preview** permet d'effectuer des recherches en temps réel et d'enrichir les réponses avec des informations actualisées.

**Caractéristiques :**
- Accès à des informations en temps réel (actualités, cours boursiers, météo, etc.)
- **Citations automatiques** : Le modèle cite ses sources
- Disponible via la **Responses API** (bêta)
- Modèles compatibles : `gpt-4o-mini`, `gpt-4o`

**Différence avec Chat Completions :**
- Responses API : Interface simplifiée avec `input` et `output`
- Support natif des outils comme web_search
- Moins de contrôle sur les paramètres avancés

**Note importante :** Cette fonctionnalité est en préversion et peut évoluer.

> **Référence** : l'enrichissement d'une réponse par recherche web en temps réel s'apparente au *Retrieval-Augmented Generation* (RAG) — Lewis et al. 2020, *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, arXiv:2005.11401.


In [6]:
# Web Search basique via Responses API
print("Recherche web en cours...\n")

response = client.responses.create(
    model=DEFAULT_MODEL,
    tools=[{"type": "web_search_preview"}],
    input="Quelles sont les dernières avancées majeures en intelligence artificielle en janvier 2026?"
)

print("=== Recherche Web : IA en 2026 ===")
for item in response.output:
    if hasattr(item, 'content'):
        print(item.content)
        print()

Recherche web en cours...



=== Recherche Web : IA en 2026 ===
[]

[]

[]

[]

[]

[]

[]

[]

[]

[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=556, start_index=445, title='NVIDIA Kicks Off the Next Generation of AI With Rubin — Six New Chips, One Incredible AI Supercomputer | NVIDIA Newsroom', type='url_citation', url='https://nvidianews.nvidia.com/news/rubin-platform-ai-supercomputer?utm_source=openai'), AnnotationURLCitation(end_index=1063, start_index=942, title='D4RT: Unified, Fast 4D Scene Reconstruction & Tracking — Google DeepMind', type='url_citation', url='https://deepmind.google/blog/d4rt-teaching-ai-to-see-the-world-in-four-dimensions/?utm_source=openai'), AnnotationURLCitation(end_index=1444, start_index=1360, title='Advancing regulatory variant effect prediction with AlphaGenome | Nature', type='url_citation', url='https://www.nature.com/articles/s41586-025-10014-0?utm_source=openai'), AnnotationURLCitation(end_index=1821, start_index=1746, title='January 2026 AI Releases: Genie 

### Interprétation de la recherche web

**Fonctionnement de `web_search_preview` :**

1. **Requête formulée** : Le modèle génère une requête de recherche optimisée
2. **Recherche effectuée** : Interrogation de sources web en temps réel
3. **Agrégation** : Synthèse des résultats multiples
4. **Citations** : Ajout automatique de références aux sources

**Avantages par rapport aux connaissances pré-entraînées :**

| Critère | Connaissances pré-entraînées | Web Search |
|---------|------------------------------|------------|
| **Actualité** | Coupure en octobre 2023 | Temps réel |
| **Précision temporelle** | Approximative | Exacte (dates, événements récents) |
| **Sources** | Implicites | Citées explicitement |
| **Fiabilité** | Haute (entraînement massif) | Variable (dépend des sources) |

**Latence observée :**

- Requête web search : **~3-5 secondes** (vs. ~1 seconde pour Chat Completions standard)
- Compromis : Actualité vs. vitesse de réponse

In [7]:
# Web Search avec données financières en temps réel
print("Recherche d'informations financières...\n")

response = client.responses.create(
    model=DEFAULT_MODEL,
    tools=[{"type": "web_search_preview"}],
    input="Quel est le cours actuel de l'action Apple (AAPL) et quelles sont ses performances sur les 3 derniers mois?"
)

print("=== Informations financières en temps réel ===")
for item in response.output:
    if hasattr(item, 'content'):
        print(item.content)
        print()

# Note: Les citations sont incluses automatiquement dans la réponse

Recherche d'informations financières...



=== Informations financières en temps réel ===
[]

[]

[]

[]

[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=372, start_index=280, title='Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance', type='url_citation', url='https://ca.finance.yahoo.com/quote/AAPL/history/?utm_source=openai'), AnnotationURLCitation(end_index=614, start_index=522, title='Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance', type='url_citation', url='https://ca.finance.yahoo.com/quote/AAPL/history/?utm_source=openai'), AnnotationURLCitation(end_index=856, start_index=764, title='Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance', type='url_citation', url='https://ca.finance.yahoo.com/quote/AAPL/history/?utm_source=openai')], text='Voici les chiffres — en valeur et en pourcentage — avec les dates précises.\n\n- Cours le plus récent disponible : 319,97 USD (dernier cours fourni par la source — 5 septembre 2026). \n\n- Cours de clôture il y a environ 3 mo

### Cas d'usage : Données financières en temps réel

**Pourquoi le web search est critique ici :**

Les **cours boursiers** changent en continu. Un modèle avec connaissances figées (octobre 2023) ne peut pas fournir :
- Le cours actuel d'une action
- Les variations récentes (3 derniers mois)
- Les événements récents affectant le titre

**Applications professionnelles :**

| Métier | Usage |
|--------|-------|
| **Traders** | Analyse rapide de titres avec contexte récent |
| **Analystes** | Recherche de tendances sectorielles actualisées |
| **Journalistes** | Vérification de données financières pour articles |
| **Conseillers** | Briefings clients avec informations jour |

**Exemple de citations attendues :**

Le modèle devrait inclure automatiquement des références comme :
- "Selon Bloomberg (4 février 2026)..."
- "D'après Yahoo Finance..."
- "Source : MarketWatch..."

> **Attention** : Les informations financières de sources web peuvent avoir quelques minutes de retard. Pour du trading haute fréquence, utiliser des API financières spécialisées (Alpha Vantage, IEX Cloud).

### Exercice 2 : Comparaison multi-sources avec web search

Utilisez le web search pour comparer les informations sur un même sujet provenant de différentes sources. Le but est de synthetiser une reponse equilibree en citant au moins 3 sources distinctes.

**Objectif** : Formuler une requête de recherche web qui demande au modèle de presenter les différents points de vue sur un sujet debattu, avec les sources pour chaque position.

**Indices** :
- Utilisez `client.responses.create()` avec `tools=[{"type": "web_search_preview"}]`
- Choisissez un sujet avec des debats actifs (ex: "Faut-il reguler les modèles d'IA generative ?")
- Demandez explicitement au modèle de "presenter les arguments pour et contre avec des sources"
- Verifiez que les annotations (`annotations`) dans la reponse citent bien des sources différentes

In [8]:
# Exercice 2 : Comparaison multi-sources avec web search
# TODO etudiant : Creez une requete de recherche web comparee

# Etape 1 : Formuler la requete avec demande de balance
# query_debat = """
# Faut-il reguler les modeles d'IA generative ?
# Presente les arguments pour et contre, en citant au moins 3 sources distinctes.
# """

# Etape 2 : Appeler la Responses API avec web search
# response_debat = client.responses.create(
#     model=DEFAULT_MODEL,
#     tools=[{"type": "web_search_preview"}],
#     input=query_debat
# )

# Etape 3 : Extraire le texte et les citations
# for item in response_debat.output:
#     if hasattr(item, 'content'):
#         for content in item.content:
#             if hasattr(content, 'text'):
#                 print(content.text)
#                 print(f"\nSources citees: {len(content.annotations)}")
#                 for ann in content.annotations:
#                     print(f"  - {ann.title}: {ann.url}")

print("Exercice a completer")

Exercice a completer


## 3. Combiner PDF et Web Search

Le véritable pouvoir vient de la **combinaison** de ces deux fonctionnalités :

**Cas d'usage :**
- **Analyse de rapports enrichie** : Comparer les données d'un rapport PDF avec les tendances actuelles
- **Fact-checking** : Vérifier des affirmations dans un document avec des sources web
- **Veille concurrentielle** : Analyser un rapport interne et le contextualiser avec l'actualité du secteur
- **Actualisation de documents** : Identifier les informations obsolètes dans un PDF

**Workflow typique :**
1. Extraire les informations clés du PDF
2. Formuler une requête web basée sur ces informations
3. Combiner les deux sources pour une analyse enrichie

In [9]:
# Workflow combiné : Document Image + Web Search
print("=== ÉTAPE 1 : Extraction des informations du document ===\n")

# Analyser le document image pour extraire les données financières.
# Budget relevé à 4000 : gpt-5-mini (modèle de raisonnement) consomme son budget de
# raisonnement dans max_completion_tokens ; à 300 l'extraction revenait vide et la
# cascade web n'avait aucun chiffre réel à contextualiser (#3571).
doc_analysis = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "text", 
                "text": "Extrais le chiffre d'affaires, le taux de croissance et les perspectives Q2 de ce rapport."
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{img_base64}"}
            }
        ]
    }],
    max_completion_tokens=4000
)

ca_info = doc_analysis.choices[0].message.content
print("Données extraites du document:")
print(ca_info)
print()

# Construire une requête web contextualisée
print("=== ÉTAPE 2 : Enrichissement avec données web ===\n")

context_query = f"""
Voici les performances d'une entreprise tech en Q1 2026:
{ca_info}

Compare ces résultats avec:
1. Les performances moyennes du secteur tech en 2026
2. Les tendances de croissance actuelles
3. Les perspectives pour Q2 2026

Fournis une analyse comparative brève.
"""

web_context = client.responses.create(
    model=DEFAULT_MODEL,
    tools=[{"type": "web_search_preview"}],
    input=context_query
)

print("=== Analyse enrichie (Document + Web) ===")
for item in web_context.output:
    if hasattr(item, 'content'):
        print(item.content)
        print()

=== ÉTAPE 1 : Extraction des informations du document ===



Données extraites du document:
- Chiffre d'affaires Q1 : 2.5 M EUR  
- Taux de croissance (Q1) : +15%  
- Perspectives Q2 : Objectif 3 M EUR (croissance prévue +20%)

=== ÉTAPE 2 : Enrichissement avec données web ===



=== Analyse enrichie (Document + Web) ===
[]

[]

[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=522, start_index=335, title='Forrester: Global Technology Spend Will Grow By 7.8% In 2026 To Reach $5.6 Trillion | Forrester Research, Inc', type='url_citation', url='https://forresterresearch.gcs-web.com/news-releases/news-release-details/forrester-global-technology-spend-will-grow-78-2026-reach-56/?utm_source=openai'), AnnotationURLCitation(end_index=973, start_index=786, title='Forrester: Global Technology Spend Will Grow By 7.8% In 2026 To Reach $5.6 Trillion | Forrester Research, Inc', type='url_citation', url='https://forresterresearch.gcs-web.com/news-releases/news-release-details/forrester-global-technology-spend-will-grow-78-2026-reach-56/?utm_source=openai'), AnnotationURLCitation(end_index=1477, start_index=1368, title='June, Second Quarter 2026 Review and Outlook | Nasdaq', type='url_citation', url='https://www.nasdaq.com/articles/june-second-quarter-2026-revie

### Interprétation du workflow combiné

**Architecture de l'analyse enrichie :**

```
Document PDF/Image
    ↓
[EXTRACTION] → Données structurées (CA, croissance, perspectives)
    ↓
[CONTEXTUALISATION] → Requête web formulée
    ↓
[WEB SEARCH] → Tendances secteur, benchmarks, actualité
    ↓
[SYNTHÈSE] → Rapport comparatif enrichi
```

L'extraction vision récupère les chiffres internes du document (CA 2,5 M EUR,
+15%, perspectives Q2 3 M EUR +20%), puis la requête web contextualise ces
résultats avec les tendances secteur 2026 (ex. prévisions de dépenses IT de
Gartner). La synthèse compare les performances internes aux benchmarks externes.

**Pourquoi cette approche est puissante :**

1. **Données internes** (PDF) : informations propriétaires, chiffres précis
2. **Contexte externe** (Web) : comparaison sectorielle, tendances marché
3. **Synthèse** : analyse comparative que ni la source PDF ni le web seuls ne peuvent fournir

**Cas d'usage concrets :**

| Scénario | Valeur ajoutée |
|----------|----------------|
| **Rapport trimestriel** | Comparer performances entreprise vs. concurrents |
| **Proposition commerciale** | Enrichir avec données marché récentes |
| **Audit** | Vérifier conformité avec réglementations actualisées |
| **Due diligence** | Croiser données fournies avec informations publiques |

**Coût indicatif du workflow :**

- **Étape 1** (Vision) : ~1958 tokens (image, usage constaté) + sortie variable
- **Étape 2** (Web Search) : ~500 tokens input + 800 tokens output
- **Total** : de l'ordre de ~3300 tokens avec `gpt-5-mini` ≈ $0.004

> **Optimisation** : pour traiter des lots de documents, utiliser `batch API` pour réduire les coûts de 50%.</cell>
>

## 4. Exemple avancé : Vérification de faits

Un autre cas d'usage puissant : vérifier les affirmations d'un document PDF avec des sources web récentes.

In [10]:
# Fact-checking : Vérifier une affirmation du document
print("=== Vérification de faits ===\n")

# Extraire une affirmation spécifique.
# Budget relevé à 4000 : à 200 l'extraction d'affirmation revenait vide, et la
# vérification web tournait sur un sujet générique au lieu d'une claim du document (#3571).
claim_extraction = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Quelle est l'affirmation principale sur les perspectives de croissance dans ce rapport?"
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{img_base64}"}
            }
        ]
    }],
    max_completion_tokens=4000
)

claim = claim_extraction.choices[0].message.content
print(f"Affirmation extraite: {claim}\n")

# Vérifier avec web search
verification_query = f"""
Affirmation à vérifier: {claim}

Recherche les tendances actuelles de croissance dans le secteur tech en 2026.
Cette affirmation est-elle réaliste? Cite des sources récentes.
"""

verification = client.responses.create(
    model=DEFAULT_MODEL,
    tools=[{"type": "web_search_preview"}],
    input=verification_query
)

print("=== Résultat de la vérification ===")
for item in verification.output:
    if hasattr(item, 'content'):
        print(item.content)

=== Vérification de faits ===



Affirmation extraite: L'affirmation principale : Perspectives Q2 — objectif 3M EUR (+20%).  
Autrement dit, viser une croissance de 20 % (passer de 2,5M EUR en Q1 à 3M EUR en Q2), soutenue par le lancement du produit Alpha, l'expansion en Europe et le recrutement de 20 ingénieurs.



=== Résultat de la vérification ===
[]
[]
[]
[]
[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=608, start_index=400, title='Gartner Forecasts Worldwide IT Spending to Grow 13.5% in 2026, Totaling $6.31 Trillion', type='url_citation', url='https://www.gartner.com/en/newsroom/press-releases/2026-04-22-gartner-forecasts-worldwide-it-spending-to-grow-13-point-5-percent-in-2026-totaling-6-point-31-trillion-dollars?utm_source=openai'), AnnotationURLCitation(end_index=1238, start_index=1134, title='SaaS operating benchmarks 2026 · Array Capital', type='url_citation', url='https://www.arraycapital.com/insights/saas-operating-benchmarks/?utm_source=openai'), AnnotationURLCitation(end_index=1769, start_index=1634, title='Product Marketing Statistics 2026: Launch & Positioning', type='url_citation', url='https://www.digitalapplied.com/blog/product-marketing-statistics-2026-launch-positioning-data?utm_source=openai'), AnnotationURLCitation(end_index=2377, start_index=2277, title=

### Interprétation du fact-checking

L'extraction isole l'affirmation principale du document (objectif CA Q2 de 3 M EUR,
+20%, porté par le lancement du produit Alpha, l'expansion européenne et le
recrutement de 20 ingénieurs), puis la vérification web confronte cette claim aux
tendances tech 2026. Le modèle produit une analyse sourcée (citations web réelles)
qui évalue le réalisme de l'affirmation au regard du contexte sectoriel.

**Méthodologie de vérification en 2 temps :**

1. **Extraction de l'affirmation** : isolation de la claim spécifique du document
2. **Recherche contradictoire** : vérification avec sources externes actualisées

**Indicateurs de fiabilité :**

| Critère | Vérification |
|---------|--------------|
| **Nombre de sources** | ≥3 sources convergentes = forte fiabilité |
| **Dates des sources** | Sources < 1 mois = très fiables |
| **Autorité** | Sources officielles (institutions, médias réputés) |
| **Cohérence** | Concordance entre sources indépendantes |

**Applications critiques :**

- **Journalism** : Vérification automatisée de communiqués de presse
- **Compliance** : Détection de déclarations non conformes dans rapports
- **Legal** : Validation de faits dans documents contractuels
- **Research** : Cross-validation de données dans publications

**Limites du fact-checking automatisé :**

⚠️ **Biais des sources web** : Les résultats de recherche peuvent privilégier certaines sources  
⚠️ **Nuances manquées** : Affirmations partiellement vraies peuvent être mal évaluées  
⚠️ **Contexte temporel** : Une affirmation vraie en 2025 peut être fausse en 2026  

> **Recommandation** : Toujours vérifier **manuellement** les citations fournies par le modèle. Le web search est un **outil d'aide**, pas un arbitre absolu de vérité.</cell>
>

### Exercice 3 : Resume de recherche web sur un sujet technique

Créez une fonction qui effectue une recherche web sur un sujet technique, extrait les points cles, et genere un resume structure avec les sources citees.

**Objectif** : Utiliser la Responses API avec `web_search_preview` pour rechercher un sujet, puis formater la reponse avec les citations.

**Indices** :
- Utilisez `client.responses.create()` avec `tools=[{"type": "web_search_preview"}]`
- Parcourez `response.output` pour trouver les items avec `hasattr(item, 'content')`
- Pour chaque `ResponseOutputText`, le champ `text` contient le texte et `annotations` les citations
- Formatez le résultat en listant : titre de la source, URL, et extrait pertinent

In [11]:
# Exercice 3 : Resume de recherche web sur un sujet technique
# TODO etudiant : Creez une fonction de recherche web avec extraction des sources

# Etape 1 : Definir la fonction de recherche
# def search_and_summarize(query: str) -> str:
#     response = client.responses.create(
#         model=DEFAULT_MODEL,
#         tools=[{"type": "web_search_preview"}],
#         input=query
#     )
#     sources = []
#     summary = ""
#     for item in response.output:
#         if hasattr(item, 'content'):
#             for content in item.content:
#                 if hasattr(content, 'text'):
#                     summary = content.text
#                     for ann in content.annotations:
#                         sources.append({"title": ann.title, "url": ann.url})
#     return summary, sources

# Etape 2 : Tester avec un sujet technique
# resume, sources = search_and_summarize(
#     "Quelles sont les differences entre RAG et fine-tuning pour les LLMs en 2026?"
# )

# Etape 3 : Afficher le resume et les sources
# print("=== Resume ===")
# print(resume)
# print("\n=== Sources ===")
# for i, src in enumerate(sources, 1):
#     print(f"{i}. {src['title']}: {src['url']}")

print("Exercice a completer")

Exercice a completer


## 5. Limitations et bonnes pratiques

### Limitations PDF

| Contrainte | Limite | Impact |
|------------|--------|--------|
| **Pages** | 100 max | Documents longs nécessitent découpage |
| **Taille** | 32 MB | PDFs avec images haute résolution peuvent dépasser |
| **Coût** | 1 page = 1 image | Un PDF de 10 pages coûte autant que 10 images |
| **Qualité OCR** | Variable | Texte dans images peut être mal reconnu |

### Bonnes pratiques PDF

1. **Optimiser les PDF** : Compresser avant envoi
2. **Découper si nécessaire** : Traiter par sections pour documents longs
3. **Privilégier le texte** : PDFs textuels > PDFs scannés
4. **Vérifier les coûts** : Calculer tokens avant traitement massif

### Limitations Web Search

- **Latence** : Requêtes web ajoutent 2-5 secondes
- **Fiabilité** : Sources web peuvent être incorrectes
- **Coût** : Requêtes web consomment plus de tokens
- **Disponibilité** : Fonctionnalité en préversion (bêta)

### Bonnes pratiques Web Search

1. **Vérifier les citations** : Toujours consulter les sources mentionnées
2. **Queries spécifiques** : Plus la requête est précise, meilleurs les résultats
3. **Combiner avec connaissances** : Ne pas tout déléguer au web search
4. **Gérer les erreurs** : Prévoir des fallbacks si la recherche échoue

### Estimation des coûts

**Exemple de calcul pour GPT-4o-mini :**
- PDF 10 pages : ~10 images × 2833 tokens = ~28,000 tokens input
- Web search : ~500-1000 tokens supplémentaires
- Total pour notre workflow : ~30,000 tokens input + 500 output
- Coût estimé : $0.045 (tarif janvier 2026)

**Recommandation :** Toujours tester avec `gpt-4o-mini` avant d'utiliser `gpt-4o` (10× plus cher).

In [12]:
# Nettoyage : Supprimer l'image de test
import os

if os.path.exists("test_report.png"):
    os.remove("test_report.png")
    print("✓ Fichier de test supprimé")
else:
    print("Aucun fichier à nettoyer")

✓ Fichier de test supprimé


## Conclusion

### Ce que nous avons appris

1. **Support PDF natif** : Les modèles vision peuvent analyser des PDFs directement
   - Encodage base64 pour envoi direct
   - Limites : 100 pages, 32 MB
   - Coût : 1 page = 1 image

2. **Web Search** : Accès en temps réel à l'information
   - Via Responses API avec `web_search_preview`
   - Citations automatiques
   - Idéal pour données actuelles

3. **Combinaison PDF + Web** : Analyses enrichies
   - Extraction de données PDF
   - Contextualisation avec sources web
   - Fact-checking et vérification

### Cas d'usage professionnels

| Domaine | Application |
|---------|-------------|
| **Finance** | Analyse de rapports avec données marché en temps réel |
| **Juridique** | Vérification de conformité avec réglementations actuelles |
| **Recherche** | Actualisation de revues de littérature |
| **Consulting** | Benchmarking clients vs. tendances secteur |
| **Journalism** | Fact-checking automatisé de documents |

### Exercices suggérés

1. **Niveau débutant** :
   - Analyser votre CV (PDF) et obtenir des conseils basés sur les tendances emploi actuelles
   - Créer un résumé enrichi d'un article de recherche

2. **Niveau intermédiaire** :
   - Développer un système de veille qui compare des rapports trimestriels successifs avec l'actualité
   - Créer un fact-checker pour articles de presse (PDF) vs. sources web

3. **Niveau avancé** :
   - Pipeline automatisé d'analyse de documents contractuels avec vérification de conformité légale
   - Système de recommandation qui analyse des rapports internes et suggère des actions basées sur les tendances marché

### Prochaines étapes

- **Notebook 7** : Structured Outputs (JSON Schema forcé)
- **Notebook 8** : Function Calling avancé
- **Notebook 9** : Assistants API et Code Interpreter

### Ressources complémentaires

- [Documentation OpenAI - Vision](https://platform.openai.com/docs/guides/vision)
- [Responses API Reference](https://platform.openai.com/docs/api-reference/responses)
- [Guide des prix OpenAI](https://openai.com/api/pricing/)